# Naive RAG: stuffing the whole document into the prompt

This notebook is the "before" picture. It answers a question using your own
document with **no chunking, no embeddings, no vector database** - just:

1. read a file into a string,
2. paste that string into the prompt,
3. ask the LLM.

That works fine for a five-paragraph file. In the last section we deliberately
try it on a much bigger "book" to see exactly where and why this approach
breaks down. The next notebook, `02_rag_ingestion_pipeline.ipynb`, builds the
proper fix (retrieval) from scratch.

**Setup:** get a free Gemini API key from https://aistudio.google.com/apikey,
then either set it as an environment variable named `GOOGLE_API_KEY`, or paste
it into the `GOOGLE_API_KEY` variable in the next cell (fine for a personal
learning notebook - never hardcode real keys in shared code).


In [1]:
# If you don't have these installed yet:
# %pip install -q google-generativeai


In [2]:
import os
import google.generativeai as genai

GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "PASTE_YOUR_API_KEY_HERE")
genai.configure(api_key=GOOGLE_API_KEY)

MODEL_NAME = "gemini-1.5-flash"  # a small, fast, cheap Gemini model
model = genai.GenerativeModel(MODEL_NAME)


C:\Users\Samir\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Create a small reference document, then "load" it

The "loader" here is nothing more than opening the file and reading it into a
string - there's no need for a framework's `DocumentLoader` class when the
file is this simple. We'll build a real, page-aware PDF loader in the next
notebook.


In [3]:
import os

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)
DOC_PATH = os.path.join(DATA_DIR, "water_cycle.md")

FIVE_PARAGRAPHS = """# The Water Cycle

The water cycle, also known as the hydrological cycle, describes the continuous movement of water on, above, and below the surface of the Earth. It has no real starting point, but for convenience we can begin with the oceans, which hold about 97 percent of the planet's water.

The first major stage is evaporation. Heat from the sun causes liquid water in oceans, lakes, and rivers to turn into water vapor, which rises into the atmosphere. Plants also release water vapor through a related process called transpiration, and together the two are often called evapotranspiration.

As water vapor rises higher into the atmosphere, it cools and condenses into tiny droplets, forming clouds. This stage is called condensation. When enough droplets gather and combine, they become too heavy to stay suspended in the air.

At that point, the water falls back to Earth as precipitation, in the form of rain, snow, sleet, or hail depending on the temperature of the surrounding air. This precipitation replenishes rivers, lakes, and groundwater, and much of it eventually flows back into the oceans through a process called collection, or runoff.

The water cycle is essential to almost every ecosystem on Earth. It regulates climate, shapes landscapes through erosion, and supplies fresh water to plants, animals, and humans. Without this constant recycling of water between the atmosphere, land, and oceans, life as we know it would not be possible.
"""

with open(DOC_PATH, "w", encoding="utf-8") as f:
    f.write(FIVE_PARAGRAPHS)

print(f"Wrote {len(FIVE_PARAGRAPHS)} characters to {DOC_PATH}")


Wrote 1464 characters to data\water_cycle.md


In [4]:
# The "loader": just read the file back into a plain string. No wrapper library needed.
def load_text_file(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

document_text = load_text_file(DOC_PATH)
print(document_text)


# The Water Cycle

The water cycle, also known as the hydrological cycle, describes the continuous movement of water on, above, and below the surface of the Earth. It has no real starting point, but for convenience we can begin with the oceans, which hold about 97 percent of the planet's water.

The first major stage is evaporation. Heat from the sun causes liquid water in oceans, lakes, and rivers to turn into water vapor, which rises into the atmosphere. Plants also release water vapor through a related process called transpiration, and together the two are often called evapotranspiration.

As water vapor rises higher into the atmosphere, it cools and condenses into tiny droplets, forming clouds. This stage is called condensation. When enough droplets gather and combine, they become too heavy to stay suspended in the air.

At that point, the water falls back to Earth as precipitation, in the form of rain, snow, sleet, or hail depending on the temperature of the surrounding air. This 

## 2. Answer a question using the raw content - no chunking, no embeddings

We use an f-string to drop the *entire* document straight into the prompt,
verbatim. The instruction "you are a helpful assistant, answer using this
reference" is the whole trick behind this naive approach.


In [5]:
question = "What are the main stages of the water cycle, and what role does the sun play in it?"

prompt = f"""You are a helpful assistant.
Answer the question using ONLY the reference text below. If the answer is not
contained in the reference, say you don't know.

Reference:
\"\"\"
{document_text}
\"\"\"

Question: {question}
"""

print(prompt)


You are a helpful assistant.
Answer the question using ONLY the reference text below. If the answer is not
contained in the reference, say you don't know.

Reference:
"""
# The Water Cycle

The water cycle, also known as the hydrological cycle, describes the continuous movement of water on, above, and below the surface of the Earth. It has no real starting point, but for convenience we can begin with the oceans, which hold about 97 percent of the planet's water.

The first major stage is evaporation. Heat from the sun causes liquid water in oceans, lakes, and rivers to turn into water vapor, which rises into the atmosphere. Plants also release water vapor through a related process called transpiration, and together the two are often called evapotranspiration.

As water vapor rises higher into the atmosphere, it cools and condenses into tiny droplets, forming clouds. This stage is called condensation. When enough droplets gather and combine, they become too heavy to stay suspended in th

In [6]:
response = model.generate_content(prompt)
print(response.text)


InvalidArgument: 400 API key not valid. Please pass a valid API key. [reason: "API_KEY_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "generativelanguage.googleapis.com"
}
, locale: "en-US"
message: "API key not valid. Please pass a valid API key."
]

## 3. Now try it with a *complete book* instead of five paragraphs

To avoid downloading anything or using copyrighted text, we generate a
synthetic "book": a few hundred chapters of unrelated filler, with our real
water-cycle paragraphs hidden in the middle as the one chapter that actually
matters (a classic "needle in a haystack" setup). Then we ask the exact same
question and look at what it costs us in tokens - and whether a smaller
("low version") Gemini model can even accept the request at all.


In [7]:
import random

FILLER_TOPICS = [
    "the history of the printing press",
    "how coral reefs form",
    "the rules of chess",
    "the migration patterns of monarch butterflies",
    "how bread is baked",
    "the structure of a jet engine",
    "the invention of the telephone",
    "how glaciers carve valleys",
    "the basics of double-entry bookkeeping",
    "the life cycle of a star",
    "how vaccines train the immune system",
    "the history of the Silk Road",
    "how suspension bridges stay up",
    "the rules of cricket",
    "how volcanoes erupt",
]

def filler_chapter(topic: str, chapter: int) -> str:
    return (
        f"Chapter {chapter} discusses {topic}. "
        f"This section explores background, key figures, and open questions "
        f"related to {topic}, with no connection at all to water or the water cycle. "
        f"Readers interested in {topic} will find several examples and a short "
        f"summary at the end of the chapter."
    )

random.seed(42)
NUM_CHAPTERS = 600
NEEDLE_CHAPTER = 314  # the one chapter that actually contains our reference material

chapters = []
for i in range(1, NUM_CHAPTERS + 1):
    if i == NEEDLE_CHAPTER:
        chapters.append(f"Chapter {i}.\n\n{document_text}")
    else:
        topic = random.choice(FILLER_TOPICS)
        chapters.append(f"Chapter {i}.\n\n{filler_chapter(topic, i)}")

book_text = "\n\n".join(chapters)
print(f"Synthetic book: {len(book_text):,} characters across {len(chapters)} chapters")
print(f"The real reference material is hidden in Chapter {NEEDLE_CHAPTER}")


Synthetic book: 213,269 characters across 600 chapters
The real reference material is hidden in Chapter 314


### Count tokens before sending anything

Every Gemini model has a maximum input size (its *context window*), measured
in tokens, not characters. Let's measure the book against that limit before
we even make a request.


In [8]:
SMALL_MODEL_NAME = "gemini-1.5-flash-8b"  # a lighter/cheaper "low version" model
# Swap this for an older small-context model such as "gemini-1.0-pro" (32,768
# token limit) if you want to force a hard failure regardless of book size.
small_model = genai.GenerativeModel(SMALL_MODEL_NAME)

reference_tokens = model.count_tokens(document_text).total_tokens
book_tokens = small_model.count_tokens(book_text).total_tokens

CONTEXT_WINDOW = 1_000_000  # documented input limit for gemini-1.5-flash-8b, in tokens

print(f"Tokens in the actual answer-relevant reference: {reference_tokens:,}")
print(f"Tokens in the whole synthetic book:              {book_tokens:,}")
print(f"{SMALL_MODEL_NAME}'s context window:             {CONTEXT_WINDOW:,}")
print(
    f"Share of the book that's irrelevant to our question: "
    f"{100 * (book_tokens - reference_tokens) / book_tokens:.4f}%"
)

if book_tokens > CONTEXT_WINDOW:
    print("\nThis exceeds the model's context window - the call below WILL fail.")
else:
    print(
        "\nThis technically fits under the context window, but look at the "
        "irrelevant-content percentage above: you are about to pay for, and "
        "wait on, an enormous amount of text just to ask about five paragraphs' "
        "worth of content. Try increasing NUM_CHAPTERS above, or switching "
        "SMALL_MODEL_NAME to an older/smaller model, to push this over the edge."
    )


InvalidArgument: 400 API key not valid. Please pass a valid API key. [reason: "API_KEY_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "generativelanguage.googleapis.com"
}
, locale: "en-US"
message: "API key not valid. Please pass a valid API key."
]

In [9]:
book_prompt = f"""You are a helpful assistant.
Answer the question using ONLY the reference text below. If the answer is not
contained in the reference, say you don't know.

Reference:
\"\"\"
{book_text}
\"\"\"

Question: {question}
"""

try:
    response = small_model.generate_content(book_prompt)
    print(response.text)
except Exception as e:
    print(
        "The request failed - this is the proof that 'just paste the whole "
        "book into the prompt' does not scale once a document gets large "
        "enough, or once you're on a smaller-context model:\n"
    )
    print(repr(e))


The request failed - this is the proof that 'just paste the whole book into the prompt' does not scale once a document gets large enough, or once you're on a smaller-context model:

InvalidArgument('API key not valid. Please pass a valid API key.')


## Takeaway

Even when the naive approach *works*, it's wasteful: you pay in tokens,
latency, and money for the entire book on every single question, when the
real answer lives in one small part of it. And once a document (or model) is
small enough on one side of that ratio, you hit a hard wall - the context
window - and the request fails outright.

The fix is **retrieval**: split the document into small chunks, find only the
handful of chunks that are actually relevant to the question, and put just
those in the prompt. That's exactly what `02_rag_ingestion_pipeline.ipynb`
builds next, from scratch.
